# Feature Engineering Experiments

**Purpose**: Test how adding or removing specific features impacts model performance. Identify features that may be hurting accuracy.

**Experiments**:
1. Baseline: Current feature set performance
2. Remove near-zero importance features (from SHAP analysis)
3. Remove each feature group systematically (ablation study)
4. Add new behavioral features and measure impact
5. Feature selection via recursive elimination

In [ ]:
import sys
sys.path.append('../..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import pickle
from datetime import datetime
from sklearn.impute import SimpleImputer
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

from src.models.fusion import weighted_late_fusion
from src.visualization.plots import set_style

np.random.seed(42)
set_style('whitegrid')

# Paths
OUTPUT_DIR = Path('../../data/results/analysis_outputs_PRE')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("="*80)
print("FEATURE ENGINEERING EXPERIMENTS")
print("="*80)
print(f"Started: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

FEATURE ENGINEERING EXPERIMENTS
Started: 2026-03-19 12:44:05


## 1. Load Data and Establish Baseline

In [ ]:
# Load extracted features
with open('../../data/results/features_PRE/extracted_features_PRE.pkl', 'rb') as f:
    feature_data = pickle.load(f)

merged_df = feature_data['merged_df']
physio_cols = feature_data['physio_cols']
behavior_cols = feature_data['behavior_cols']
gaze_cols = feature_data['gaze_cols']

print(f"Total trials: {len(merged_df)}")
print(f"Total subjects: {merged_df['subject_id'].nunique()}")
print(f"\nFeature counts:")
print(f"  Physiology: {len(physio_cols)}")
print(f"  Behavior: {len(behavior_cols)}")
print(f"  Gaze: {len(gaze_cols)}")
print(f"  Total: {len(physio_cols) + len(behavior_cols) + len(gaze_cols)}")

print(f"\nPhysio features: {physio_cols}")
print(f"\nBehavior features: {behavior_cols}")
print(f"\nGaze features: {gaze_cols}")

Total trials: 12511
Total subjects: 97

Feature counts:
  Physiology: 13
  Behavior: 7
  Gaze: 20
  Total: 40

Physio features: ['pupil_mean_pre', 'pupil_std_pre', 'pupil_slope_pre', 'time_to_peak_pre', 'pupil_cv_pre', 'pupil_velocity_mean_pre', 'pupil_max_dilation_rate_pre', 'pupil_max_constriction_rate_pre', 'pupil_acceleration_std_pre', 'pct_time_dilating_pre', 'num_dilation_peaks_pre', 'eye_asymmetry_pre', 'eye_asymmetry_std_pre']

Behavior features: ['reaction_time', 'decision_time', 'ev_difference', 'invest_variance', 'ambiguity', 'condition_social', 'risk_premium']

Gaze features: ['gaze_valid_pct', 'gaze_x_mean', 'gaze_x_std', 'gaze_y_mean', 'gaze_y_std', 'screen_x_mean', 'screen_x_std', 'screen_y_mean', 'screen_y_std', 'gaze_velocity_mean', 'gaze_velocity_std', 'gaze_velocity_max', 'gaze_acceleration_mean', 'gaze_acceleration_std', 'fixation_ratio', 'saccade_ratio', 'saccade_count', 'gaze_dispersion_x', 'gaze_dispersion_y', 'gaze_path_length']


In [ ]:
# Load SHAP importance for reference
shap_df = pd.read_csv(OUTPUT_DIR / 'shap_importance_all_PRE.csv')
print("\nSHAP Feature Importance (sorted):")
print(shap_df.sort_values('mean_abs_shap', ascending=False).head(20).to_string(index=False))


SHAP Feature Importance (sorted):
                        feature  mean_abs_shap
                      ambiguity       0.063962
                  reaction_time       0.030013
                pupil_slope_pre       0.020291
                  pupil_std_pre       0.014243
                  ev_difference       0.010396
                invest_variance       0.009702
                   risk_premium       0.009355
          pct_time_dilating_pre       0.007996
               time_to_peak_pre       0.005593
                  screen_y_mean       0.005344
                   pupil_cv_pre       0.003586
                    gaze_y_mean       0.003314
        pupil_velocity_mean_pre       0.002701
                     gaze_x_std       0.002085
    pupil_max_dilation_rate_pre       0.001991
                 pupil_mean_pre       0.001865
                    gaze_x_mean       0.001694
               gaze_path_length       0.001666
          gaze_acceleration_std       0.001657
pupil_max_constriction_ra

In [ ]:
def run_experiment(df, physio_cols, behavior_cols, gaze_cols, experiment_name):
    """
    Run late fusion model with specified feature sets and return results.
    """
    # Prepare features
    X_physio = SimpleImputer(strategy='mean').fit_transform(df[physio_cols]) if len(physio_cols) > 0 else np.zeros((len(df), 1))
    X_behavior = SimpleImputer(strategy='mean').fit_transform(df[behavior_cols]) if len(behavior_cols) > 0 else np.zeros((len(df), 1))
    X_gaze = SimpleImputer(strategy='mean').fit_transform(df[gaze_cols]) if len(gaze_cols) > 0 else np.zeros((len(df), 1))
    
    y = df['outcome'].values
    subjects = df['subject_id'].values
    
    # Run model
    X_modalities = [X_physio, X_behavior, X_gaze]
    modality_names = ['Physiology', 'Behavior', 'Gaze']
    
    results = weighted_late_fusion(
        X_modalities, y, subjects, modality_names,
        fusion_method='weighted'
    )
    
    return {
        'experiment': experiment_name,
        'accuracy': results['accuracy_mean'],
        'accuracy_sem': results['accuracy_sem'],
        'f1_score': results['f1_mean'],
        'f1_sem': results['f1_sem'],
        'physio_weight': results['weights'][0],
        'behavior_weight': results['weights'][1],
        'gaze_weight': results['weights'][2],
        'n_physio': len(physio_cols),
        'n_behavior': len(behavior_cols),
        'n_gaze': len(gaze_cols),
        'n_total': len(physio_cols) + len(behavior_cols) + len(gaze_cols)
    }

# Storage for all experiment results
all_results = []

In [ ]:
# Experiment 1: BASELINE
print("\n" + "="*80)
print("EXPERIMENT 1: BASELINE (All current features)")
print("="*80)

baseline_result = run_experiment(merged_df, physio_cols, behavior_cols, gaze_cols, "Baseline")
all_results.append(baseline_result)

print(f"\nAccuracy: {baseline_result['accuracy']:.4f} ± {baseline_result['accuracy_sem']:.4f}")
print(f"F1-Score: {baseline_result['f1_score']:.4f} ± {baseline_result['f1_sem']:.4f}")
print(f"Total features: {baseline_result['n_total']}")

BASELINE_ACC = baseline_result['accuracy']


EXPERIMENT 1: BASELINE (All current features)

Accuracy: 0.6931 ± 0.0133
F1-Score: 0.6794 ± 0.0157
Total features: 40


## 2. Remove Near-Zero Importance Features

In [ ]:
print("\n" + "="*80)
print("EXPERIMENT 2: Remove Near-Zero SHAP Importance Features")
print("="*80)

# Identify near-zero importance features (SHAP < 0.001)
near_zero_features = shap_df[shap_df['mean_abs_shap'] < 0.001]['feature'].tolist()
print(f"\nNear-zero importance features ({len(near_zero_features)}):")
for f in near_zero_features:
    shap_val = shap_df[shap_df['feature'] == f]['mean_abs_shap'].values[0]
    print(f"  - {f}: {shap_val:.6f}")

# Remove from each feature set
physio_cols_filtered = [c for c in physio_cols if c not in near_zero_features]
behavior_cols_filtered = [c for c in behavior_cols if c not in near_zero_features]
gaze_cols_filtered = [c for c in gaze_cols if c not in near_zero_features]

print(f"\nFeatures removed:")
print(f"  From physio: {set(physio_cols) - set(physio_cols_filtered)}")
print(f"  From behavior: {set(behavior_cols) - set(behavior_cols_filtered)}")
print(f"  From gaze: {set(gaze_cols) - set(gaze_cols_filtered)}")

result = run_experiment(merged_df, physio_cols_filtered, behavior_cols_filtered, gaze_cols_filtered, 
                        "Remove near-zero SHAP")
all_results.append(result)

diff = result['accuracy'] - BASELINE_ACC
print(f"\nAccuracy: {result['accuracy']:.4f} ± {result['accuracy_sem']:.4f}")
print(f"Change from baseline: {diff:+.4f} ({100*diff/BASELINE_ACC:+.2f}%)")
print(f"Features removed: {baseline_result['n_total'] - result['n_total']}")


EXPERIMENT 2: Remove Near-Zero SHAP Importance Features

Near-zero importance features (12):
  - pupil_acceleration_std_pre: 0.000964
  - screen_y_std: 0.000829
  - screen_x_mean: 0.000724
  - screen_x_std: 0.000705
  - gaze_velocity_std: 0.000684
  - gaze_velocity_max: 0.000666
  - gaze_velocity_mean: 0.000607
  - condition_social: 0.000083
  - fixation_ratio: 0.000077
  - saccade_ratio: 0.000075
  - saccade_count: 0.000034
  - gaze_valid_pct: 0.000000

Features removed:
  From physio: {'pupil_acceleration_std_pre'}
  From behavior: {'condition_social'}
  From gaze: {'screen_y_std', 'gaze_velocity_mean', 'screen_x_mean', 'screen_x_std', 'gaze_valid_pct', 'gaze_velocity_max', 'fixation_ratio', 'gaze_velocity_std', 'saccade_ratio', 'saccade_count'}

Accuracy: 0.6948 ± 0.0134
Change from baseline: +0.0017 (+0.24%)
Features removed: 12


In [ ]:
# Experiment 2b: More aggressive removal (SHAP < 0.01)
print("\n" + "-"*80)
print("EXPERIMENT 2b: Remove Low SHAP Features (< 0.01)")
print("-"*80)

low_importance_features = shap_df[shap_df['mean_abs_shap'] < 0.01]['feature'].tolist()
print(f"\nLow importance features ({len(low_importance_features)}):")
for f in low_importance_features:
    shap_val = shap_df[shap_df['feature'] == f]['mean_abs_shap'].values[0]
    print(f"  - {f}: {shap_val:.6f}")

physio_cols_v2 = [c for c in physio_cols if c not in low_importance_features]
behavior_cols_v2 = [c for c in behavior_cols if c not in low_importance_features]
gaze_cols_v2 = [c for c in gaze_cols if c not in low_importance_features]

result = run_experiment(merged_df, physio_cols_v2, behavior_cols_v2, gaze_cols_v2,
                        "Remove low SHAP (<0.01)")
all_results.append(result)

diff = result['accuracy'] - BASELINE_ACC
print(f"\nAccuracy: {result['accuracy']:.4f} ± {result['accuracy_sem']:.4f}")
print(f"Change from baseline: {diff:+.4f} ({100*diff/BASELINE_ACC:+.2f}%)")
print(f"Features removed: {baseline_result['n_total'] - result['n_total']}")


--------------------------------------------------------------------------------
EXPERIMENT 2b: Remove Low SHAP Features (< 0.01)
--------------------------------------------------------------------------------

Low importance features (34):
  - invest_variance: 0.009702
  - risk_premium: 0.009355
  - pct_time_dilating_pre: 0.007996
  - time_to_peak_pre: 0.005593
  - screen_y_mean: 0.005344
  - pupil_cv_pre: 0.003586
  - gaze_y_mean: 0.003314
  - pupil_velocity_mean_pre: 0.002701
  - gaze_x_std: 0.002085
  - pupil_max_dilation_rate_pre: 0.001991
  - pupil_mean_pre: 0.001865
  - gaze_x_mean: 0.001694
  - gaze_path_length: 0.001666
  - gaze_acceleration_std: 0.001657
  - pupil_max_constriction_rate_pre: 0.001546
  - gaze_y_std: 0.001460
  - num_dilation_peaks_pre: 0.001436
  - eye_asymmetry_std_pre: 0.001080
  - eye_asymmetry_pre: 0.001068
  - gaze_acceleration_mean: 0.001064
  - gaze_dispersion_y: 0.001009
  - gaze_dispersion_x: 0.001003
  - pupil_acceleration_std_pre: 0.000964
  - scr

## 3. Modality Ablation Study

In [ ]:
print("\n" + "="*80)
print("EXPERIMENT 3: Modality Ablation Study")
print("="*80)

# 3a: Remove all physiology features
print("\n--- 3a: Without Physiology ---")
result = run_experiment(merged_df, [], behavior_cols, gaze_cols, "No Physiology")
all_results.append(result)
diff = result['accuracy'] - BASELINE_ACC
print(f"Accuracy: {result['accuracy']:.4f} (change: {diff:+.4f})")

# 3b: Remove all gaze features
print("\n--- 3b: Without Gaze ---")
result = run_experiment(merged_df, physio_cols, behavior_cols, [], "No Gaze")
all_results.append(result)
diff = result['accuracy'] - BASELINE_ACC
print(f"Accuracy: {result['accuracy']:.4f} (change: {diff:+.4f})")

# 3c: Remove all behavior features
print("\n--- 3c: Without Behavior ---")
result = run_experiment(merged_df, physio_cols, [], gaze_cols, "No Behavior")
all_results.append(result)
diff = result['accuracy'] - BASELINE_ACC
print(f"Accuracy: {result['accuracy']:.4f} (change: {diff:+.4f})")

# 3d: Behavior only
print("\n--- 3d: Behavior Only ---")
result = run_experiment(merged_df, [], behavior_cols, [], "Behavior Only")
all_results.append(result)
diff = result['accuracy'] - BASELINE_ACC
print(f"Accuracy: {result['accuracy']:.4f} (change: {diff:+.4f})")

# 3e: Physiology only
print("\n--- 3e: Physiology Only ---")
result = run_experiment(merged_df, physio_cols, [], [], "Physiology Only")
all_results.append(result)
diff = result['accuracy'] - BASELINE_ACC
print(f"Accuracy: {result['accuracy']:.4f} (change: {diff:+.4f})")

# 3f: Gaze only
print("\n--- 3f: Gaze Only ---")
result = run_experiment(merged_df, [], [], gaze_cols, "Gaze Only")
all_results.append(result)
diff = result['accuracy'] - BASELINE_ACC
print(f"Accuracy: {result['accuracy']:.4f} (change: {diff:+.4f})")


EXPERIMENT 3: Modality Ablation Study

--- 3a: Without Physiology ---
Accuracy: 0.6958 (change: +0.0028)

--- 3b: Without Gaze ---
Accuracy: 0.6928 (change: -0.0003)

--- 3c: Without Behavior ---
Accuracy: 0.6559 (change: -0.0372)

--- 3d: Behavior Only ---
Accuracy: 0.6959 (change: +0.0029)

--- 3e: Physiology Only ---
Accuracy: 0.6562 (change: -0.0369)

--- 3f: Gaze Only ---
Accuracy: 0.6566 (change: -0.0364)


## 4. Individual Feature Removal (Leave-One-Out)

In [ ]:
print("\n" + "="*80)
print("EXPERIMENT 4: Leave-One-Feature-Out Analysis")
print("="*80)
print("Testing if removing individual features IMPROVES accuracy...")

loo_results = []

# Test removing each gaze feature (most likely to hurt performance)
print("\n--- Testing Gaze Features ---")
for feat in gaze_cols:
    gaze_cols_minus_one = [c for c in gaze_cols if c != feat]
    result = run_experiment(merged_df, physio_cols, behavior_cols, gaze_cols_minus_one, f"Remove {feat}")
    diff = result['accuracy'] - BASELINE_ACC
    loo_results.append({
        'feature_removed': feat,
        'modality': 'gaze',
        'accuracy': result['accuracy'],
        'change': diff,
        'pct_change': 100 * diff / BASELINE_ACC
    })
    if diff > 0.001:
        print(f"  {feat}: {result['accuracy']:.4f} ({diff:+.4f}) ** IMPROVES **")
    elif diff < -0.001:
        print(f"  {feat}: {result['accuracy']:.4f} ({diff:+.4f}) -- hurts --")
    else:
        print(f"  {feat}: {result['accuracy']:.4f} ({diff:+.4f})")

# Test removing each physio feature
print("\n--- Testing Physiology Features ---")
for feat in physio_cols:
    physio_cols_minus_one = [c for c in physio_cols if c != feat]
    result = run_experiment(merged_df, physio_cols_minus_one, behavior_cols, gaze_cols, f"Remove {feat}")
    diff = result['accuracy'] - BASELINE_ACC
    loo_results.append({
        'feature_removed': feat,
        'modality': 'physio',
        'accuracy': result['accuracy'],
        'change': diff,
        'pct_change': 100 * diff / BASELINE_ACC
    })
    if diff > 0.001:
        print(f"  {feat}: {result['accuracy']:.4f} ({diff:+.4f}) ** IMPROVES **")
    elif diff < -0.001:
        print(f"  {feat}: {result['accuracy']:.4f} ({diff:+.4f}) -- hurts --")
    else:
        print(f"  {feat}: {result['accuracy']:.4f} ({diff:+.4f})")


EXPERIMENT 4: Leave-One-Feature-Out Analysis
Testing if removing individual features IMPROVES accuracy...

--- Testing Gaze Features ---
  gaze_valid_pct: 0.6928 (-0.0003)
  gaze_x_mean: 0.6937 (+0.0006)
  gaze_x_std: 0.6935 (+0.0004)


In [ ]:
# Summarize LOO results
loo_df = pd.DataFrame(loo_results).sort_values('change', ascending=False)

print("\n" + "-"*80)
print("LEAVE-ONE-OUT SUMMARY")
print("-"*80)

print("\nFeatures that IMPROVE accuracy when removed (potential harmful features):")
improvers = loo_df[loo_df['change'] > 0.0005]
if len(improvers) > 0:
    print(improvers[['feature_removed', 'modality', 'accuracy', 'change', 'pct_change']].to_string(index=False))
else:
    print("  None found")

print("\nFeatures that HURT accuracy when removed (valuable features):")
hurters = loo_df[loo_df['change'] < -0.001]
if len(hurters) > 0:
    print(hurters[['feature_removed', 'modality', 'accuracy', 'change', 'pct_change']].to_string(index=False))
else:
    print("  None found")

## 5. Remove Multiple Harmful Features

In [ ]:
print("\n" + "="*80)
print("EXPERIMENT 5: Remove All Potentially Harmful Features")
print("="*80)

# Identify features that improved accuracy when removed
harmful_features = loo_df[loo_df['change'] > 0]['feature_removed'].tolist()

if len(harmful_features) > 0:
    print(f"\nPotentially harmful features identified: {harmful_features}")
    
    physio_cols_clean = [c for c in physio_cols if c not in harmful_features]
    behavior_cols_clean = [c for c in behavior_cols if c not in harmful_features]
    gaze_cols_clean = [c for c in gaze_cols if c not in harmful_features]
    
    result = run_experiment(merged_df, physio_cols_clean, behavior_cols_clean, gaze_cols_clean,
                            "Remove all harmful")
    all_results.append(result)
    
    diff = result['accuracy'] - BASELINE_ACC
    print(f"\nAccuracy: {result['accuracy']:.4f} ± {result['accuracy_sem']:.4f}")
    print(f"Change from baseline: {diff:+.4f} ({100*diff/BASELINE_ACC:+.2f}%)")
    print(f"Features removed: {len(harmful_features)}")
else:
    print("\nNo harmful features identified - all features contribute positively or neutrally.")

## 6. Test Specific Feature Groups

In [ ]:
print("\n" + "="*80)
print("EXPERIMENT 6: Test Removing Specific Feature Groups")
print("="*80)

# 6a: Remove all saccade-related features
saccade_features = [c for c in gaze_cols if 'saccade' in c.lower()]
print(f"\n--- 6a: Remove saccade features ({saccade_features}) ---")
gaze_no_saccade = [c for c in gaze_cols if c not in saccade_features]
result = run_experiment(merged_df, physio_cols, behavior_cols, gaze_no_saccade, "No saccade features")
all_results.append(result)
diff = result['accuracy'] - BASELINE_ACC
print(f"Accuracy: {result['accuracy']:.4f} (change: {diff:+.4f})")

# 6b: Remove all fixation-related features
fixation_features = [c for c in gaze_cols if 'fixation' in c.lower()]
print(f"\n--- 6b: Remove fixation features ({fixation_features}) ---")
gaze_no_fixation = [c for c in gaze_cols if c not in fixation_features]
result = run_experiment(merged_df, physio_cols, behavior_cols, gaze_no_fixation, "No fixation features")
all_results.append(result)
diff = result['accuracy'] - BASELINE_ACC
print(f"Accuracy: {result['accuracy']:.4f} (change: {diff:+.4f})")

# 6c: Remove velocity/acceleration features
velocity_features = [c for c in gaze_cols if 'velocity' in c.lower() or 'acceleration' in c.lower()]
print(f"\n--- 6c: Remove velocity/acceleration features ({velocity_features}) ---")
gaze_no_velocity = [c for c in gaze_cols if c not in velocity_features]
result = run_experiment(merged_df, physio_cols, behavior_cols, gaze_no_velocity, "No velocity features")
all_results.append(result)
diff = result['accuracy'] - BASELINE_ACC
print(f"Accuracy: {result['accuracy']:.4f} (change: {diff:+.4f})")

# 6d: Keep only position features (screen_x, screen_y, gaze_x, gaze_y means)
position_features = [c for c in gaze_cols if ('screen_x' in c or 'screen_y' in c or 'gaze_x' in c or 'gaze_y' in c) and 'mean' in c]
print(f"\n--- 6d: Keep only position mean features ({position_features}) ---")
result = run_experiment(merged_df, physio_cols, behavior_cols, position_features, "Position means only")
all_results.append(result)
diff = result['accuracy'] - BASELINE_ACC
print(f"Accuracy: {result['accuracy']:.4f} (change: {diff:+.4f})")

## 7. Results Summary

In [ ]:
# Create results summary
results_df = pd.DataFrame(all_results)
results_df['change_from_baseline'] = results_df['accuracy'] - BASELINE_ACC
results_df['pct_change'] = 100 * results_df['change_from_baseline'] / BASELINE_ACC
results_df = results_df.sort_values('accuracy', ascending=False)

print("\n" + "="*80)
print("ALL EXPERIMENT RESULTS (sorted by accuracy)")
print("="*80)

display_cols = ['experiment', 'accuracy', 'accuracy_sem', 'change_from_baseline', 'pct_change', 'n_total']
print(results_df[display_cols].to_string(index=False))

In [ ]:
# Visualize results
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Sort by accuracy for plotting
plot_df = results_df.sort_values('accuracy', ascending=True)

# 1. Accuracy comparison
ax1 = axes[0]
colors = ['green' if x > 0 else 'red' if x < 0 else 'gray' for x in plot_df['change_from_baseline']]
bars = ax1.barh(range(len(plot_df)), plot_df['accuracy'], color=colors, alpha=0.7, edgecolor='black')
ax1.axvline(x=BASELINE_ACC, color='blue', linestyle='--', linewidth=2, label=f'Baseline ({BASELINE_ACC:.4f})')
ax1.axvline(x=0.5, color='gray', linestyle=':', linewidth=1, label='Chance')
ax1.set_yticks(range(len(plot_df)))
ax1.set_yticklabels(plot_df['experiment'])
ax1.set_xlabel('Accuracy', fontsize=11, fontweight='bold')
ax1.set_title('Accuracy by Feature Configuration', fontsize=12, fontweight='bold')
ax1.legend(loc='lower right')
ax1.grid(True, alpha=0.3, axis='x')

# 2. Change from baseline
ax2 = axes[1]
colors = ['green' if x > 0 else 'red' for x in plot_df['change_from_baseline']]
ax2.barh(range(len(plot_df)), plot_df['change_from_baseline'], color=colors, alpha=0.7, edgecolor='black')
ax2.axvline(x=0, color='black', linestyle='-', linewidth=1)
ax2.set_yticks(range(len(plot_df)))
ax2.set_yticklabels(plot_df['experiment'])
ax2.set_xlabel('Change from Baseline', fontsize=11, fontweight='bold')
ax2.set_title('Impact of Feature Changes', fontsize=12, fontweight='bold')
ax2.grid(True, alpha=0.3, axis='x')

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'feature_engineering_experiments.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Save results
results_df.to_csv(OUTPUT_DIR / 'feature_engineering_experiments.csv', index=False)
loo_df.to_csv(OUTPUT_DIR / 'leave_one_out_feature_analysis.csv', index=False)

print("\n" + "="*80)
print("KEY FINDINGS")
print("="*80)

best_config = results_df.iloc[0]
worst_config = results_df.iloc[-1]

print(f"""
1. BASELINE ACCURACY: {BASELINE_ACC:.4f}

2. BEST CONFIGURATION:
   - {best_config['experiment']}
   - Accuracy: {best_config['accuracy']:.4f}
   - Change: {best_config['change_from_baseline']:+.4f} ({best_config['pct_change']:+.2f}%)

3. WORST CONFIGURATION:
   - {worst_config['experiment']}
   - Accuracy: {worst_config['accuracy']:.4f}
   - Change: {worst_config['change_from_baseline']:+.4f} ({worst_config['pct_change']:+.2f}%)

4. FEATURES THAT HURT PERFORMANCE (removing them helps):
""")

improvers = loo_df[loo_df['change'] > 0].sort_values('change', ascending=False)
if len(improvers) > 0:
    for _, row in improvers.iterrows():
        print(f"   - {row['feature_removed']} ({row['modality']}): +{row['change']:.4f}")
else:
    print("   None identified")

print(f"""
5. RECOMMENDATIONS:
   - Consider removing features with negative impact
   - Behavior features are critical (removing them causes largest drop)
   - Gaze features contribute minimally
""")

print(f"\nResults saved to {OUTPUT_DIR}/")
print(f"Analysis complete: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")